In [1]:
import os
import json
from openai import OpenAI
import gradio as gr
from dotenv import load_dotenv
import sqlite3

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [3]:
system_message = """You are a helpful assistant for an Airline called FLightAI. 
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
 """

In [7]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)")
    conn.commit()

In [8]:
def get_ticket_estimate(destination_city):
    print(f"DB Tool called for city: {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price from prices where city = ?", (destination_city.lower(),))
        result = cursor.fetchone()

        return f"The price of a ticket to {destination_city} is {result[0]}" if result else "No price data available for this city"

In [12]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("INSERT INTO prices (city, price) values (?,?) ON CONFLICT(city) DO UPDATE SET price = ?", (city.lower(), price, price))
        conn.commit()

In [13]:
ticket_prices = {"London":"$799", "Paris":"$501", "Tokyo":"$1500"}

for city, price in ticket_prices.items():
    set_ticket_price(city, price)


get_ticket_estimate("London")

DB Tool called for city: London


'The price of a ticket to London is $799'

In [ ]:
price_function = {
    "name":"get_ticket_estimate",
    "description":"Get the price of a return ticket to the destination city.",
    "parameters":{
        "type":"object",
        "properties":{
            "destination_city":{
                "type":"string",
                "description":"The city the customer wants to travel to."
            }
        },
        "required":["destination_city"],
        "additionalProperties":False
    }
}

In [ ]:
tools = [{"type":"function", "function":price_function}]

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_estimate":
            arguments = json.loads(tool_call.function.arguments)
            print("Arguments: ", arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_estimate(city)
            responses.append({
                "role":"tool",
                "content":price_details,
                "tool_call_id":tool_call.id
            })
    return responses

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role":"system", "content": system_message}]\
                     + history\
                         + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(
    fn=chat
).launch()